# Lakewood, Colorado — Land Value Tax Policy Analysis

**City of Lakewood**, Jefferson County (Census FIPS **08059**), Colorado's fifth-largest
city. A Lakewood property-tax bill is a *stack* of levies from many independent taxing
entities. This model touches **only the City of Lakewood's own municipal mill levy** and
holds every other entity's levy unchanged.

## Policy decisions (Greg, 2026-08-20)

| Parameter | Setting |
|---|---|
| Levy scope | **City of Lakewood's own mill levy only** — 4.7110 mills |
| Reform | Revenue-neutral **split-rate**, land taxed **4x** improvements |
| Exemptions | **Preserve all** — fully-exempt parcels held out of the solver, export and charts |
| Geographic scope | Parcels whose representative point falls inside Lakewood city limits |

### Held UNCHANGED (every other entity in the stack)

Jefferson County, Jeffco Public Schools R-1, West Metro Fire Protection District,
Foothills Park & Recreation District, Urban Drainage & Flood Control (Mile High Flood
District), RTD, Law Enforcement Authority, and all metro districts, sanitation
districts, business improvement districts and urban renewal authorities.

## ⚠️ The scope caveat that governs every number in this notebook

Lakewood's own levy is **4.7110 mills of a ~91.4-mill stack — about 5.2% of a Lakewood
property-tax bill.** Every percentage in this notebook is a percentage **of the city's
own share**, not of the total bill. A parcel shown as "-13%" sees its *city* tax fall
13%, which is roughly 0.7% of its total property-tax bill. Dollar magnitudes are
correspondingly small: the whole city levy is $15.8M, and the median single-family home's
city tax is on the order of $170/year.

This is a real, council-controlled revenue stream and the redistribution within it is
genuine — but no slide may say "your property taxes fall X%" without naming the city-only
scope.

## Colorado's assessment machinery

Colorado taxes an *assessed* value derived from the assessor's *actual* (market) value,
and post-Gallagher the assessment rate **differs between school-district levies and other
local-government levies**. Lakewood's own levy is a **non-school local-government levy**,
so the *local* rate is the correct basis.

Jefferson County's parcel file publishes the assessor's own `ASMASDLND` / `ASMASDIMP`
assessed land/improvement split alongside `TOTACTLNDV` / `TOTACTIMPV` actual values, so
this model **does not assume a statutory assessment rate** — it uses the assessor's
arithmetic directly. Section 3 nevertheless reverse-engineers the statutory formula from
those columns as a data-integrity check and to date the roll year.

## The roll-year mismatch, stated plainly

The parcel file is the **roll-2026** assessment; the most recent **certified** levy is
**roll 2025**. Section 6 therefore uses the repo's Baltimore/Madison derived-millage
pattern: derive the municipal rate that makes modeled revenue equal the certified levy,
and report published rate, derived rate, and the difference. A revenue-neutral split-rate
result is *invariant* to the level of the municipal millage — scaling the millage scales
every parcel's current and new tax by the same constant — so the mismatch moves dollar
magnitudes only, never a percentage.

## Data sources

| What | Source |
|---|---|
| Parcels + assessor values + tax district + structure detail | Jefferson County jMap `Parcel` MapServer layer 0 (258,941 county features, 236 fields) — `https://mapservices2.jeffco.us/arcgis/rest/services/jMap/Parcel/MapServer/0` |
| City limits | Census TIGER 2025 Incorporated Places, CO place GEOID **0843000** (Lakewood city) |
| Certified mill levy | Jeffco Assessor, *2025 Tax Authorities with Mill Levy*, entity code **3070 LAKEWOOD = 4.7110** |
| Certified valuation + levy | Jeffco Assessor, *2025 Final Certification of Value* (Dec 4 2025), entity 3070; Colorado DPT 55th Annual Report |
| Statutory assessment rates | HB24B-1001 (enrolled) |
| Demographics | Census ACS 2022 block groups, county FIPS 08059 |

## Section 1 — Imports, constants, `.env`

In [ ]:
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, str(Path("../..").resolve()))
REPO_ROOT = Path("../..").resolve()
load_dotenv(REPO_ROOT / ".env")          # CENSUS_API_KEY is NOT auto-loaded

from lvt.lvt_utils import (
    calculate_category_tax_summary,
    calculate_current_tax,
    model_split_rate_tax,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.policy_analysis import analyze_land_by_improvement_share

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

CITY_NAME = "lakewood"
STATE_FIPS = "08"
COUNTY_FIPS = "059"                  # Jefferson County, CO
LAND_IMPROVEMENT_RATIO = 4.0
MODEL_TYPE = "split_rate:4.0"

# ---- Roll years ----
ROLL_YEAR_PARCELS = 2026             # proven in Section 3 by fitting the statutory formula
ROLL_YEAR_LEVY = 2025                # most recent certified levy

# ---- Official figures: City of Lakewood's OWN levy, tax year 2025 ----
# Jeffco Assessor, "2025 Tax Authorities with Mill Levy" -> code 3070 LAKEWOOD 4.7110
# Jeffco Assessor, "2025 Final Certification of Value" (Dec 4, 2025), tax entity 3070
LAKEWOOD_MILLS_PUBLISHED = 4.7110
OFFICIAL_CITY_LEVY = 15_801_930.0            # 4.7110 mills x gross AV -- the certified levy
LAKEWOOD_GROSS_AV_2025 = 3_354_262_401.0     # current year's GROSS total taxable assessed valuation
LAKEWOOD_TIF_INCREMENT_2025 = 113_990_024.0  # less total TIF area increments
LAKEWOOD_NET_AV_2025 = 3_240_272_377.0       # current year's NET total taxable assessed valuation
CITY_CASH_LEVY_NET_TIF = 15_264_923.0        # 4.7110 mills x net AV -- what reaches the city
LAKEWOOD_ACTUAL_REAL_2025 = 38_801_814_016.0 # total actual value of all real property (incl. exempt religious/school/charitable)
LAKEWOOD_ACTUAL_TAXABLE_2025 = 38_584_774_831.0  # total actual value of all TAXABLE property
TYPICAL_STACK_MILLS = 91.399                 # most common Lakewood tax-tag total levy

# Lakewood's levy is 100% General Fund: no bond redemption, no contractual obligation,
# no override and no abatement mills. TY2023 and TY2024 carried a temporary mill levy
# rate reduction (net 4.280 and 4.496); TY2025 has none, so gross == net == 4.7110.

# ---- Parcel-map identity columns ----
PARCEL_ID_COL = "PIN"
OWNER_NAME_COL = "OWNNAM"
OWNER_ADDRESS_COL = "PRPADDRESS"
# Jefferson County's assessor portal is session-based; no stable per-parcel deep link
# exists (tested property.jeffco.us, propertysearch.jeffco.us, gis.jeffco.us -> 404).
PARCEL_URL_TEMPLATE = None

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PARCEL_PATH = DATA_DIR / "parcels_raw.gpq"
PLACE_PATH = DATA_DIR / "lakewood_place.parquet"

PARCEL_LAYER = "https://mapservices2.jeffco.us/arcgis/rest/services/jMap/Parcel/MapServer/0"
PLACE_LAYER = ("https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
               "Places_CouSub_ConCity_SubMCD/MapServer/4")
PLACE_GEOID = "0843000"               # Lakewood city, CO

print(f"City: {CITY_NAME}   FIPS: {STATE_FIPS + COUNTY_FIPS}   Model: {MODEL_TYPE}")
print(f"CENSUS_API_KEY present: {bool(os.getenv('CENSUS_API_KEY'))}")
print(f"Lakewood's own levy: {LAKEWOOD_MILLS_PUBLISHED} mills of ~{TYPICAL_STACK_MILLS} "
      f"= {LAKEWOOD_MILLS_PUBLISHED / TYPICAL_STACK_MILLS * 100:.1f}% of a tax bill")

## Section 2 — Load parcels and restrict to Lakewood city limits

Jefferson County's `Parcel` layer is countywide (258,941 features). The download is
restricted to an envelope around Lakewood to keep the pull tractable, then parcels are
cut to the city by **representative-point-in-polygon** against the Census TIGER 2025
incorporated-place boundary for Lakewood city (GEOID `0843000`).

The envelope is deliberately larger than the city, so it also contains Golden, Wheat
Ridge, Edgewater, Morrison, Littleton, Arvada and unincorporated Jefferson County —
the point-in-polygon step is what removes them, and the cell below verifies it did.

Fetched as **geoJSON** so holes and multi-part parcels survive.

In [ ]:
PARCEL_FIELDS = ["OBJECTID","PIN","SCH","AIN","OWNNAM","PRPADDRESS","PRPCTYNAM","SUBNAM",
    "NHDNBR","NHDNAM","POLYCAT","ATD","TTD","MILL_LEVY","GENBIL","GENPD",
    "TAXCLS","VALACT","TAXCLS2","VALACT2","TAXCLS3","VALACT3",
    "TOTACTVAL","TOTACTLNDV","TOTACTIMPV","PYRTOTVAL",
    "ASMASDTOT","ASMASDLND","ASMASDIMP","ASMASDPRS",
    "LGLSQFT","TOTACR","STTSTRC","STTTYPUSE","STTYRBLT","STTGRSAREA","STTGRSARE2",
    "STTNBRUNT","STTNBRBLDG","STTNBRFLR","MOREVAL"]

# Envelope around Lakewood (deliberately generous; the city cut happens below)
ENVELOPE = {"geometry": "-105.21,39.61,-105.045,39.77",
            "geometryType": "esriGeometryEnvelope", "inSR": "4326",
            "spatialRel": "esriSpatialRelIntersects"}


def _fetch_page(url, params, attempt=0):
    try:
        r = requests.get(url, params=params, timeout=420)
        r.raise_for_status()
        js = r.json()
        if "features" not in js:
            raise RuntimeError(f"no features: {str(js)[:200]}")
        return js["features"]
    except Exception:
        if attempt >= 4:
            raise
        time.sleep(4 * (attempt + 1))
        return _fetch_page(url, params, attempt + 1)


def fetch_arcgis_geojson(layer_url, fields, where="1=1", page=2000, workers=5,
                         extra=None):
    """Paginated parallel geoJSON download of an ArcGIS layer."""
    q = f"{layer_url}/query"
    extra = extra or {}
    n = requests.get(q, params={"where": where, "returnCountOnly": "true", "f": "json",
                                **extra}, timeout=180).json()["count"]
    offsets = list(range(0, n, page))
    print(f"  {n:,} features -> {len(offsets)} pages")

    def one(off):
        return off, _fetch_page(q, {
            "where": where, "outFields": ",".join(fields), "returnGeometry": "true",
            "orderByFields": "OBJECTID", "resultOffset": off, "resultRecordCount": page,
            "outSR": 4326, "geometryPrecision": 7, "f": "geojson", **extra})

    pages = {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(one, o) for o in offsets]):
            off, feats = fut.result()
            pages[off] = feats
    feats = [f for o in offsets for f in pages[o]]
    gdf = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
    gdf["geometry"] = gdf.geometry.make_valid()
    return gdf


if PARCEL_PATH.exists():
    gdf_raw = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf_raw):,} parcels from cache ({PARCEL_PATH})")
else:
    print("Downloading Jefferson County parcels in the Lakewood envelope...")
    gdf_raw = fetch_arcgis_geojson(PARCEL_LAYER, PARCEL_FIELDS, extra=ENVELOPE)
    gdf_raw.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf_raw):,} parcels")

print(f"CRS: {gdf_raw.crs}   null geometry: {gdf_raw.geometry.isna().sum():,}")
print(f"Duplicate PIN: {gdf_raw['PIN'].duplicated().sum():,}")

In [ ]:
# Lakewood city limits — Census TIGER 2025 incorporated place, GEOID 0843000
if PLACE_PATH.exists():
    place = gpd.read_parquet(PLACE_PATH)
else:
    _r = requests.get(f"{PLACE_LAYER}/query", params={
        "where": f"GEOID='{PLACE_GEOID}'", "outFields": "GEOID,NAME",
        "returnGeometry": "true", "outSR": 4326, "geometryPrecision": 7,
        "f": "geojson"}, timeout=180)
    _r.raise_for_status()
    place = gpd.GeoDataFrame.from_features(_r.json()["features"], crs="EPSG:4326")
    place["geometry"] = place.geometry.make_valid()
    place.to_parquet(PLACE_PATH)

assert len(place) == 1 and place.geometry.is_valid.all(), "bad place polygon"
print(f"Place: {place['NAME'].iloc[0]} (GEOID {place['GEOID'].iloc[0]})")
print(f"Area: {place.to_crs(2877).area.sum() / 27_878_400:.2f} sq mi")

# Representative-point-in-polygon (EPSG:2877 = NAD83 / Colorado Central ftUS)
_place_p = place.to_crs(2877)[["geometry"]]
_g = gdf_raw.to_crs(2877)
_pts = gpd.GeoDataFrame(gdf_raw[["OBJECTID"]].copy(),
                        geometry=_g.geometry.representative_point(), crs=2877)
_j = gpd.sjoin(_pts, _place_p, how="inner", predicate="within")
_in_city = set(_j["OBJECTID"].astype(int))

gdf = gdf_raw[gdf_raw["OBJECTID"].isin(_in_city)].copy().reset_index(drop=True)
print(f"\nParcels inside Lakewood city limits: {len(gdf):,} of {len(gdf_raw):,} in envelope")

In [ ]:
# Sanity check: are the neighbouring municipalities actually gone?
_kept = gdf["PRPCTYNAM"].fillna("(null)").value_counts()
_dropped = (gdf_raw[~gdf_raw["OBJECTID"].isin(_in_city)]["PRPCTYNAM"]
            .fillna("(null)").value_counts())
_cmp = pd.DataFrame({"kept_in_city": _kept, "dropped": _dropped}).fillna(0).astype(int)
_cmp["pct_kept"] = (_cmp.kept_in_city / (_cmp.kept_in_city + _cmp.dropped) * 100).round(1)
print("property-city field, in-city vs dropped:")
print(_cmp.sort_values("kept_in_city", ascending=False).head(14).to_string())

NEIGHBOURS = ["GOLDEN", "WHEAT RIDGE", "EDGEWATER", "MORRISON", "ARVADA",
              "LITTLETON", "DENVER", "LAKESIDE", "MOUNTAIN VIEW"]
print("\nneighbouring-municipality leakage into the modeled set:")
for _n in NEIGHBOURS:
    _c = int(_kept.get(_n, 0))
    print(f"  {_n:<14} {_c:>6,} parcels"
          + ("   <-- CHECK" if _c > 250 else ""))
_leak = sum(int(_kept.get(_n, 0)) for _n in NEIGHBOURS)
print(f"\ntotal leakage: {_leak:,} ({_leak / len(gdf) * 100:.2f}% of modeled parcels)")
print(f"share of modeled parcels whose property-city IS Lakewood: "
      f"{_kept.get('LAKEWOOD', 0) / len(gdf) * 100:.1f}%")
print("\nNOTE: the property-city field is an address attribute, not a jurisdiction "
      "attribute, so exact agreement is not expected; the boundary polygon governs.")

In [ ]:
# Numeric coercion
_num = ["VALACT","VALACT2","VALACT3","TOTACTVAL","TOTACTLNDV","TOTACTIMPV","PYRTOTVAL",
        "ASMASDTOT","ASMASDLND","ASMASDIMP","ASMASDPRS","LGLSQFT","TOTACR",
        "STTGRSAREA","STTGRSARE2","STTNBRUNT","STTNBRBLDG","STTNBRFLR","STTYRBLT",
        "MILL_LEVY","GENBIL","GENPD"]
for _c in _num:
    gdf[_c] = pd.to_numeric(gdf[_c], errors="coerce")

NON_PARCEL_PINS = {"ROW", "WATER", "CNTY_DNVR", "CNTY_JEFF", "NONE"}
n0 = len(gdf)
gdf = gdf[gdf["PIN"].notna()
          & ~gdf["PIN"].astype(str).str.strip().str.upper().isin(NON_PARCEL_PINS)].copy()
print(f"dropped {n0 - len(gdf):,} right-of-way / water / non-parcel rows "
      f"(PIN in {sorted(NON_PARCEL_PINS)} or null)")

n1 = len(gdf)
gdf = gdf[gdf["TOTACTVAL"].notna() & (gdf["TOTACTVAL"] > 0)].copy().reset_index(drop=True)
print(f"dropped {n1 - len(gdf):,} rows with no actual value at all "
      "(common-element polygons, unvalued slivers)")

# --- Multi-part parcels: one PIN, several geoJSON features ---
# Jefferson County returns each physical part of a multi-part parcel as its own
# feature, and every part repeats the parcel's FULL attribute row -- so the value
# columns are identical across parts, not shares of a whole. Summing them would
# double-count the parcel. The correct dedup is therefore: keep ONE attribute row
# per PIN and UNION the parts' geometries so the parcel keeps its true footprint.
_dups = gdf["PIN"].duplicated(keep=False)
if _dups.any():
    _dup_pins = gdf.loc[_dups, "PIN"].nunique()
    _identical = (gdf[_dups].groupby("PIN")["TOTACTVAL"].nunique() == 1).mean()
    print(f"\nmulti-part parcels: {int(_dups.sum()):,} rows across {_dup_pins:,} PINs")
    print(f"  share of those PINs whose parts carry an IDENTICAL actual value: "
          f"{_identical * 100:.1f}%  (confirms repeated attributes, not split shares)")
    print(f"  value that would be double-counted if summed: "
          f"${gdf.loc[_dups, 'TOTACTVAL'].sum() - gdf[_dups].groupby('PIN')['TOTACTVAL'].first().sum():,.0f}")
    _geom = gdf.loc[_dups].groupby("PIN")["geometry"].apply(
        lambda s: s.union_all() if hasattr(s, "union_all") else s.unary_union)
    gdf = gdf.drop_duplicates(subset="PIN", keep="first").copy()
    gdf.loc[gdf["PIN"].isin(_geom.index), "geometry"] = (
        gdf.loc[gdf["PIN"].isin(_geom.index), "PIN"].map(_geom).to_numpy())
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
    gdf["geometry"] = gdf.geometry.make_valid()
    print(f"  -> deduped to one row per PIN, geometries unioned")

gdf = gdf.reset_index(drop=True)
assert not gdf["PIN"].duplicated().any(), "duplicate PIN survived dedup"
print(f"\nmodeled parcel universe: {len(gdf):,} (one row per PIN)")

for _c in ["TOTACTLNDV", "TOTACTIMPV", "ASMASDLND", "ASMASDIMP", "ASMASDPRS"]:
    gdf[_c] = gdf[_c].fillna(0.0).clip(lower=0)

# Identity checks on the assessor's own arithmetic
_chk_act = (gdf.TOTACTLNDV + gdf.TOTACTIMPV - gdf.TOTACTVAL).abs()
_chk_asd = (gdf.ASMASDLND + gdf.ASMASDIMP - gdf.ASMASDTOT).abs()
print(f"\nidentity: actual land+imp == total    -> {(_chk_act > 1).sum():,} mismatches")
print(f"identity: assessed land+imp == total -> {(_chk_asd > 1).sum():,} mismatches")
print(f"assessed columns null: {gdf['ASMASDTOT'].isna().sum():,}")
print(f"parcels carrying personal property (ASMASDPRS>0): {(gdf.ASMASDPRS > 0).sum():,} "
      f"(${gdf.ASMASDPRS.sum():,.0f} assessed) -- excluded, this is a real-property model")

## Section 3 — Reverse-engineer Colorado's assessment formula from the assessor's own numbers

Colorado's assessment rate is statutory but has moved repeatedly post-Gallagher, and it
differs between school and non-school levies. Rather than *assume* a rate, fit the formula
to the parcels where Jefferson County published both the actual value and the assessed
value, and report the exact-match share.

Candidate forms:

* `assessed = round(rate x actual)` — a flat rate.
* `assessed = round(rate x (actual - min(pct x actual, cap)))` — a rate applied after a
  percentage actual-value reduction with a dollar cap, the shape Colorado's recent reform
  bills use.

A fit is trusted only when it reproduces the assessor's published number for essentially
every parcel in the class. The winning combination also **dates the roll year**: 6.8%
residential-local after a 10%/$70,000 reduction, with 25% improved commercial and 26%
other non-residential, is unique to tax year 2026 under HB24B-1001.

In [ ]:
def best_formula(actual, assessed, cands):
    best = (0.0, None)
    for rate, pct, cap in cands:
        red = np.minimum(pct * actual, cap) if cap else np.zeros_like(actual, dtype=float)
        pred = np.round((actual - red) * rate, 0)
        score = float((np.abs(pred - assessed) <= 25).mean())
        if score > best[0]:
            best = (score, (rate, pct, cap))
    return best


RES_CANDS = [(r, p, c) for r in [0.0680, 0.0670, 0.0695, 0.0705, 0.0715, 0.0725, 0.0750]
             for (p, c) in [(0.0, 0), (0.10, 70_000), (0.10, 55_000), (0.10, 80_000)]]
NON_CANDS = [(r, 0.0, 0) for r in [0.25, 0.26, 0.27, 0.2725, 0.29, 0.2163, 0.10]]
CANDS = RES_CANDS + NON_CANDS

obs = gdf[(gdf.ASMASDTOT > 0) & (gdf.TOTACTVAL > 0)].copy()
obs["cls2"] = obs["TAXCLS"].astype(str).str[:2]

rows = []
for cls, sub in obs.groupby("cls2"):
    if len(sub) < 25:
        continue
    sc, f = best_formula(sub["TOTACTVAL"].to_numpy(float),
                         sub["ASMASDTOT"].to_numpy(float), CANDS)
    rows.append({"TAXCLS_2": cls, "n": len(sub), "rate": f[0], "reduction_pct": f[1],
                 "reduction_cap": f[2], "exact_match": round(sc, 4),
                 "actual_value": sub["TOTACTVAL"].sum()})
fit = pd.DataFrame(rows).sort_values("actual_value", ascending=False)
print("=== fitted statutory formula by assessor class group (local / non-school basis) ===")
print(fit.to_string(index=False))

In [ ]:
# Consolidate: the residential fit (class group 11) is the one that dates the roll
_res = obs[obs.cls2 == "11"]
_sc, _f = best_formula(_res["TOTACTVAL"].to_numpy(float),
                       _res["ASMASDTOT"].to_numpy(float), CANDS)
RES_RATE, RES_RED_PCT, RES_RED_CAP = _f
print(f"RESIDENTIAL, local (non-school) levies: {RES_RATE:.4%} of actual value after a "
      f"{RES_RED_PCT:.0%} reduction capped at ${RES_RED_CAP:,.0f}")
print(f"  exact match on {_sc * 100:.2f}% of {len(_res):,} parcels")

_nonres = fit[fit.rate >= 0.20]
print("\nNON-RESIDENTIAL rates observed:")
print(_nonres.groupby("rate").agg(class_groups=("TAXCLS_2", "count"),
                                  parcels=("n", "sum"),
                                  actual_value=("actual_value", "sum")).to_string())

_expected = (abs(RES_RATE - 0.068) < 1e-9 and RES_RED_CAP == 70_000
             and set(_nonres.rate.round(2)) <= {0.25, 0.26})
print(f"\nroll-year signature matches tax year 2026 (HB24B-1001): {_expected}")
print(f"  -> ROLL_YEAR_PARCELS = {ROLL_YEAR_PARCELS}, certified levy is roll {ROLL_YEAR_LEVY}")
print("\nOverall effective assessment ratio: "
      f"{gdf.ASMASDTOT.sum() / gdf.TOTACTVAL.sum() * 100:.2f}%")

## Section 4 — Exemptions and the taxable base

Colorado assessor class codes beginning with `9` are the exempt classes. A parcel is held
out of the revenue-neutral solver when it has no taxable assessed value at all, or when it
carries an exempt class code and was billed nothing.

**Limitation, stated up front.** This extract has no separable senior-homestead or
disabled-veteran exemption column. Colorado's senior exemption (50% of the first $200,000
of actual value) is state-reimbursed, and the assessor's published assessed value is net
of it, so those parcels' taxable value is already reduced in this data. The model
therefore preserves the exemption exactly as the assessor applied it, but cannot report it
separately.

Personal property (`ASMASDPRS`) is excluded throughout — this is a real-property model,
and a land-value tax does not reach personal property.

In [ ]:
gdf["exempt_class"] = gdf["TAXCLS"].astype(str).str.startswith("9")
gdf["asd_land"] = gdf["ASMASDLND"].astype(float)
gdf["asd_bldg"] = gdf["ASMASDIMP"].astype(float)
gdf["asd_total"] = gdf["asd_land"] + gdf["asd_bldg"]
gdf["genbil"] = gdf["GENBIL"].fillna(0.0)

gdf["full_exmp"] = ((gdf["asd_total"] <= 0)
                    | (gdf["exempt_class"] & (gdf["genbil"] <= 0))).astype(int)

print("exempt-class x billed crosstab:")
print(pd.crosstab(gdf["exempt_class"], gdf["genbil"] > 0,
                  rownames=["TAXCLS starts 9"], colnames=["was billed"]).to_string())
print(f"\nfully-exempt parcels held out: {int(gdf.full_exmp.sum()):,}")
print(f"  of which exempt class code:  {int((gdf.full_exmp == 1).mul(gdf.exempt_class).sum()):,}")
print(f"  of which zero assessed value: {int(((gdf.full_exmp == 1) & (gdf.asd_total <= 0)).sum()):,}")
print(f"taxable parcels:             {int((gdf.full_exmp == 0).sum()):,}")
print(f"\nexempt actual value held out: ${gdf.loc[gdf.full_exmp == 1, 'TOTACTVAL'].sum():,.0f}")
print(f"taxable actual value:         ${gdf.loc[gdf.full_exmp == 0, 'TOTACTVAL'].sum():,.0f}")
print(f"taxable assessed value:       ${gdf.loc[gdf.full_exmp == 0, 'asd_total'].sum():,.0f}")

### Section 4b — Gate-5 fix: condominium units carry no land value

**This is a data artifact that had to be repaired before the model could be trusted.**

Jefferson County books **condominium units as improvement-only, with `$0` land value** —
and, unlike Denver, the land is *not* carried on a sibling or common-element parcel. Six
large Lakewood condo subdivisions were traced parcel-by-parcel (Snowbird I/II/III,
Wellington, Green Gables, Capistrano Court): **zero** of roughly 1,050 parcels carry any
land value, and the general-common-element polygons are valued at `$0` land and `$0`
improvement. The condo land value is simply absent from the roll.

Meanwhile *townhomes* in the same neighbourhoods — the closest physical analogue, also
attached housing on shared land — **do** get land allocated, at a flat per-unit figure
(e.g. every unit in Newland Park Townhomes carries exactly $86,250 of land).

### Why this must be fixed

Under a split-rate, a parcel with `$0` land pays only the improvement rate, so **every**
improvement-only parcel lands at the *identical* maximum tax cut regardless of what or
where it is. That is precisely the Gate-5 canary — "distinct property types pinned at the
same extreme value" — and it would do two things to the result: manufacture a spurious
"condos win biggest" headline, and understate the city's land base, biasing the solved
land millage upward.

### The repair

The unit *totals* are correct — they are what the county actually bills — so **the total
is held fixed and only the land/improvement split is imputed**, from the median land share
of comparable parcels *with* a land value in the same assessor neighbourhood and use
group. This is the mirror image of Denver's missing-buildings fix (there, land was right
and improvement was missing) and follows the repo's St. Paul condo precedent.

**Note on what this does and does not move.** Because the repair redistributes *within*
each parcel's assessed total, the **taxable base is unchanged** — so base validation is
identical before and after, and that is the honest result, not an omission. What changes
materially is the **land share of the base**, which is the quantity the split-rate solver
actually depends on. Both are reported below.

In [ ]:
def use_group(sttstrc, taxcls):
    """Coarse comparable-pool key for imputing a land/improvement split."""
    s = str(sttstrc).strip().upper() if pd.notna(sttstrc) else ""
    c = str(taxcls).strip() if pd.notna(taxcls) else ""
    if any(t in s for t in ("TOWNHOME", "ROWHOUSE", "CONDO", "DUPLEX", "TRIPLEX",
                            "FOURPLEX", "APARTMENT", "SENIOR", "PATIO HOME")):
        return "res_attached"
    if "SINGLE FAMILY" in s or c.startswith("11"):
        return "res_detached"
    if c.startswith("3") or any(t in s for t in ("WAREHOUSE", "INDUSTRIAL", "R&D",
                                                 "FLEX", "MANUFACTUR")):
        return "industrial"
    if c.startswith("2") or any(t in s for t in ("RETAIL", "OFFICE", "RESTAURANT",
                                                 "STORE", "MOTEL", "HOTEL", "BANK",
                                                 "AUTO", "SHOPPING", "MARKET")):
        return "commercial"
    return "other"


gdf["use_group"] = [use_group(a, b) for a, b in zip(gdf["STTSTRC"], gdf["TAXCLS"])]
# Condominium units take the attached-residential comparable pool.
gdf.loc[gdf["STTSTRC"].astype(str).str.upper().str.contains("CONDO", na=False)
        & gdf["TAXCLS"].astype(str).str.startswith("12"), "use_group"] = "res_attached"

gdf["improvement_only"] = (gdf.TOTACTLNDV <= 0) & (gdf.TOTACTIMPV > 0)

print(f"improvement-only parcels (no land value anywhere): {int(gdf.improvement_only.sum()):,}")
print(f"  actual improvement value on them: ${gdf.loc[gdf.improvement_only, 'TOTACTIMPV'].sum():,.0f}")
print(f"  = {gdf.loc[gdf.improvement_only, 'TOTACTVAL'].sum() / gdf.TOTACTVAL.sum() * 100:.2f}% "
      "of modeled actual value")
print("\nby structure type:")
print(gdf[gdf.improvement_only].groupby("STTSTRC").size()
      .sort_values(ascending=False).head(12).to_string())
print("\nby assessor class:")
print(gdf[gdf.improvement_only].groupby("TAXCLS").size()
      .sort_values(ascending=False).head(8).to_string())

In [ ]:
# Comparable land shares, from parcels that HAVE both a land and an improvement value
clean = gdf[(gdf.TOTACTLNDV > 0) & (gdf.TOTACTIMPV > 0) & (gdf.TOTACTVAL > 0)].copy()
clean["land_share"] = clean.TOTACTLNDV / clean.TOTACTVAL

by_nb_grp = clean.groupby(["NHDNAM", "use_group"])["land_share"].agg(["median", "size"])
by_nb_grp = by_nb_grp[by_nb_grp["size"] >= 8]["median"]
by_grp = clean.groupby("use_group")["land_share"].median()
GLOBAL_SHARE = float(clean["land_share"].median())

print("median land share of value, by use group (comparable pool):")
print(clean.groupby("use_group")["land_share"].agg(
    n="size", median="median", q25=lambda s: s.quantile(.25),
    q75=lambda s: s.quantile(.75)).round(3).to_string())
print(f"\nglobal median land share: {GLOBAL_SHARE:.3f}")

fix = gdf.index[gdf.improvement_only]
shares = pd.Series(
    [by_nb_grp.get((nb, gp), by_grp.get(gp, GLOBAL_SHARE))
     for nb, gp in zip(gdf.loc[fix, "NHDNAM"], gdf.loc[fix, "use_group"])],
    index=fix, dtype=float).fillna(GLOBAL_SHARE).clip(0.05, 0.80)
print(f"\nimputed land share on {len(fix):,} parcels: median {shares.median():.3f}, "
      f"IQR {shares.quantile(.25):.3f}-{shares.quantile(.75):.3f}")

In [ ]:
# Apply the repair. Totals held fixed; only the land/improvement split changes.
pre_land_actual = gdf.TOTACTLNDV.sum()
pre_asd_total = gdf.asd_total.sum()
pre_taxable_base = gdf.loc[gdf.full_exmp == 0, "asd_total"].sum()
pre_land_share_base = (gdf.loc[gdf.full_exmp == 0, "asd_land"].sum()
                       / gdf.loc[gdf.full_exmp == 0, "asd_total"].sum())

_s = shares.to_numpy()
gdf.loc[fix, "TOTACTLNDV"] = np.round(gdf.loc[fix, "TOTACTVAL"].to_numpy() * _s, 0)
gdf.loc[fix, "TOTACTIMPV"] = (gdf.loc[fix, "TOTACTVAL"].to_numpy()
                              - gdf.loc[fix, "TOTACTLNDV"].to_numpy())
gdf.loc[fix, "asd_land"] = np.round(gdf.loc[fix, "asd_total"].to_numpy() * _s, 0)
gdf.loc[fix, "asd_bldg"] = (gdf.loc[fix, "asd_total"].to_numpy()
                            - gdf.loc[fix, "asd_land"].to_numpy())

post_taxable_base = gdf.loc[gdf.full_exmp == 0, "asd_total"].sum()
post_land_share_base = (gdf.loc[gdf.full_exmp == 0, "asd_land"].sum()
                        / gdf.loc[gdf.full_exmp == 0, "asd_total"].sum())

print("=== effect of the condo land-share repair ===")
print(f"actual land value:      ${pre_land_actual:,.0f} -> ${gdf.TOTACTLNDV.sum():,.0f}"
      f"   (+${gdf.TOTACTLNDV.sum() - pre_land_actual:,.0f})")
print(f"assessed total:         ${pre_asd_total:,.0f} -> ${gdf.asd_total.sum():,.0f}"
      "   (held fixed by construction)")
print(f"taxable assessed base:  ${pre_taxable_base:,.0f} -> ${post_taxable_base:,.0f}")
print(f"LAND SHARE of the taxable base: {pre_land_share_base * 100:.2f}% -> "
      f"{post_land_share_base * 100:.2f}%   <-- the quantity the solver depends on")

# Base validation, before and after -- unchanged, and here is the proof
for _label, _base in [("before repair", pre_taxable_base), ("after repair", post_taxable_base)]:
    print(f"  base validation vs certified roll-{ROLL_YEAR_LEVY} gross AV, {_label}: "
          f"{(_base / LAKEWOOD_GROSS_AV_2025 - 1) * 100:+.2f}%")

gdf["taxable_land_value"] = gdf["asd_land"].clip(lower=0)
gdf["taxable_improvement_value"] = gdf["asd_bldg"].clip(lower=0)
gdf["taxable_total"] = gdf.taxable_land_value + gdf.taxable_improvement_value
gdf.loc[gdf.full_exmp == 1, ["taxable_land_value", "taxable_improvement_value",
                             "taxable_total"]] = 0.0
assert (gdf.taxable_total >= 0).all()

In [ ]:
# The other Gate-5 canary: a recorded structure but a placeholder improvement value
_has_struct = (gdf.STTGRSAREA.fillna(0) > 0) | (gdf.STTNBRUNT.fillna(0) > 0)
_placeholder = (gdf.TOTACTIMPV.between(1, 1000)) & _has_struct & (gdf.TOTACTLNDV > 0)
print(f"parcels with a recorded structure but improvement value <= $1,000: "
      f"{int(_placeholder.sum()):,}")
if _placeholder.sum():
    print(f"  land value on them: ${gdf.loc[_placeholder, 'TOTACTLNDV'].sum():,.0f} "
          f"({gdf.loc[_placeholder, 'TOTACTLNDV'].sum() / gdf.TOTACTLNDV.sum() * 100:.3f}%"
          " of modeled land value)")
    print(gdf[_placeholder].groupby("STTSTRC").size().sort_values(ascending=False)
          .head(8).to_string())
print("\n-> Denver's roll carried 4,363 such parcels and needed a repair; the Jefferson "
      "County roll does not have that defect at material scale (see the count above).")
gdf["placeholder_improvement"] = _placeholder

## Section 5 — Property categories

Jefferson County's `STTSTRC` structure-type field carries ~90 distinct descriptions —
rich enough to split condominiums, townhomes/rowhouses, duplexes, apartment tiers, hotels
and the commercial subtypes rather than collapsing everything into
residential/commercial. `TAXCLS` (the Colorado abstract class) supplies the fallback,
where the leading digit is the class family: `0` vacant, `1` residential, `2` commercial,
`3` industrial, `4` agricultural, `9` exempt.

**Surface parking is not separately coded** in this roll, so `Transportation - Parking`
here means purpose-built parking structures only. Any surface-parking figure for the deck
must come from a separate source, not from this classification.

In [ ]:
def categorize(sttstrc, taxcls, imp_value):
    s = str(sttstrc).strip().upper() if pd.notna(sttstrc) else ""
    c = str(taxcls).strip() if pd.notna(taxcls) else ""

    if "CONDO, RES" in s or ("CONDO" in s and c.startswith("12")):
        return "Condominium"
    if "TOWNHOME" in s and "APARTMENT" not in s:
        return "Townhome / Rowhouse"
    if "ROWHOUSE" in s:
        return "Townhome / Rowhouse"
    if "SINGLE FAMILY" in s:
        return "Single Family Residential"
    if any(t in s for t in ("DUPLEX", "TRIPLEX", "FOURPLEX", "TWO FAMILY",
                            "THREE FAMILY", "FOUR FAMILY")):
        return "Small Multi-Family (2-4 units)"
    if "APARTMENT" in s or s.startswith("SENIOR") or "MULTI FAMILY" in s:
        return "Large Multi-Family (5+ units)"
    if any(t in s for t in ("MANUFACTURED", "MOBILE", "TRAILER")):
        return "Other Residential"
    if any(t in s for t in ("ELDERLY CARE", "NURSING", "GROUP HOME", "ASSISTED",
                            "DORMITORY", "RES:")):
        return "Other Residential"

    if "PARKING" in s:
        return "Transportation - Parking"
    if "MOTEL" in s or "HOTEL" in s:
        return "Hotel"
    if any(t in s for t in ("OFFICE", "MEDICAL")):
        return "Office / Commercial Condo"
    if any(t in s for t in ("RETAIL", "SHOPPING", "STORE", "MARKET", "RESTAURANT",
                            "BANK", "AUTO", "CAR WASH", "SERVICE STATION",
                            "SUPERMARKET", "PHARMAC", "DAY CARE", "VET CLINIC",
                            "HEALTH CLUB", "THEATER", "BOWLING", "FUNERAL")):
        return "Retail / General Commercial"
    if any(t in s for t in ("WAREHOUSE", "INDUSTRIAL", "R&D", "FLEX", "MANUFACTUR",
                            "LUMBER", "TRUCK", "MINI-WAREHOUSE", "STORAGE")):
        return "Industrial"
    if any(t in s for t in ("CHURCH", "SCHOOL", "GOVERNMENT", "HOSPITAL", "LIBRARY",
                            "FIRE", "MUNICIPAL", "FRATERNAL", "CEMETERY", "CLUB",
                            "SUPPORT BLDG", "UTILITY", "COMMUNITY")):
        return "Other"

    # Fall back to the Colorado abstract class family
    if c.startswith("0"):
        return "Vacant Land"
    if c.startswith("11") or c.startswith("12"):
        return "Other Residential"
    if c.startswith("2"):
        return "Other Commercial"
    if c.startswith("3"):
        return "Industrial"
    if c.startswith("4") or c.startswith("5"):
        return "Agricultural"
    return "Other"


gdf["PROPERTY_CATEGORY"] = [categorize(a, b, c) for a, b, c in
                            zip(gdf["STTSTRC"], gdf["TAXCLS"], gdf["TOTACTIMPV"])]

# Repo-wide override: any parcel with $0 improvement value is Vacant Land, whatever
# the assessor's use code says. Applied on the ORIGINAL improvement value, so the
# condo repair in Section 4b cannot turn a genuinely vacant lot into something else.
_orig_imp_zero = gdf["improvement_only"].eq(False) & (gdf["TOTACTIMPV"] <= 0)
gdf.loc[_orig_imp_zero, "PROPERTY_CATEGORY"] = "Vacant Land"

print(gdf.groupby("PROPERTY_CATEGORY").agg(
    parcels=("PIN", "size"),
    actual_value=("TOTACTVAL", "sum"),
    taxable_assessed=("taxable_total", "sum"),
    pct_exempt=("full_exmp", "mean"),
).sort_values("parcels", ascending=False).to_string())
assert gdf["PROPERTY_CATEGORY"].notna().all()
print(f"\nunmapped structure types falling through to a class-family default: "
      f"{int((gdf.PROPERTY_CATEGORY.isin(['Other', 'Other Commercial', 'Other Residential'])).sum()):,}")

## Section 6 — Rebuild the current city tax and gate it

Lakewood's own mill levy is **uniform across the city** — the many distinct tax-tag codes
in the parcel file differ because of *other* overlapping entities (fire, park & rec,
drainage, metro and sanitation districts, BIDs), not because Lakewood's own rate varies.
So the current municipal tax is simply `taxable assessed value x Lakewood's own mills`.

### TIF basis — stated explicitly

Lakewood's certified valuation carries a **$113,990,024 TIF increment** diverted to the
Lakewood Reinvestment Authority (gross AV $3,354,262,401, net AV $3,240,272,377). The LRA
is material, not a rounding error.

**This model solves on the GROSS basis**, and the reason is like-for-like: the modeled
parcel universe *includes* the parcels sitting in urban-renewal areas, so it must be
anchored to the levy struck on the whole base — `gross AV x 4.7110 mills = $15,801,930`.
TIF is a downstream *distribution* of that money between the city's general fund and the
LRA; the taxpayer's bill is identical either way, and this model is about how the bill is
distributed across parcels. The city's own general-fund cash take is smaller
(~$15.26M net of the increment) and is reported for context but is **not** the solver
target.

### Three gates, run in order

1. **Base validation on ACTUAL value — the gate that can actually fail.** Actual value is
   unaffected by Colorado's rate changes, so this isolates whether the parcel universe and
   the city boundary cut are right.
2. **Assessed-base comparison.** Expected to come in *below* the certified roll-2025
   figure for two known reasons, both reported: the roll-2026 statutory rate cut, and
   personal property + state-assessed property, which sit in the certified levy base but
   do not exist in a real-property parcel file (`ASMASDPRS` is populated on exactly one
   parcel).
3. **Independent cross-check from actual bills.** Every parcel carries `GENBIL` (dollars
   actually billed) and `MILL_LEVY` (its total tag levy), so `GENBIL / MILL_LEVY x 1000`
   recovers the parcel's roll-2025 assessed value without using any of the modeling
   arithmetic. Summing the city's 4.7110-mill share of that is a wholly independent
   estimate of Lakewood's certified levy.

In [ ]:
# --- Gate 1: base validation on ACTUAL value ---
nonexempt = gdf.full_exmp == 0
modeled_actual = gdf.loc[nonexempt, "TOTACTVAL"].sum()
base_gap = (modeled_actual / LAKEWOOD_ACTUAL_TAXABLE_2025 - 1) * 100
print("GATE 1 - tax base validation (ACTUAL / market value)")
print(f"  modeled non-exempt actual value: ${modeled_actual:,.0f}")
print(f"  certified total actual value of all taxable property (roll {ROLL_YEAR_LEVY}): "
      f"${LAKEWOOD_ACTUAL_TAXABLE_2025:,.0f}")
print(f"  gap: {base_gap:+.2f}%   (must be within +/-5%)")
assert abs(base_gap) < 5.0, f"BASE GATE FAILED: {base_gap:.2f}%"
print("  -> PASS. The city-boundary cut and exemption handling are sound.")

# --- Gate 2: assessed-base comparison ---
taxable_base = gdf.loc[nonexempt, "taxable_total"].sum()
print(f"\nGATE 2 - assessed base comparison")
print(f"  modeled taxable assessed base (roll {ROLL_YEAR_PARCELS}): ${taxable_base:,.0f}")
print(f"  certified GROSS assessed valuation (roll {ROLL_YEAR_LEVY}):  "
      f"${LAKEWOOD_GROSS_AV_2025:,.0f}")
print(f"  difference: {(taxable_base / LAKEWOOD_GROSS_AV_2025 - 1) * 100:+.2f}%")
print("  expected and in the expected direction. Two causes, neither an error:")
print(f"   (a) roll-{ROLL_YEAR_PARCELS} statutory rate cut: residential 6.95% -> 6.80%, "
      "non-residential 27% -> 25%/26%")
print(f"   (b) personal property + state-assessed property are in the certified base but "
      f"not in a real-property parcel file (ASMASDPRS populated on "
      f"{int((gdf.ASMASDPRS > 0).sum())} parcel(s))")

In [ ]:
# --- Gate 3: independent cross-check from the parcels' actual tax bills ---
_m = gdf["MILL_LEVY"].fillna(0)
gdf["implied_av_2025"] = np.where(_m > 0, gdf["genbil"] / _m * 1000.0, 0.0)
_billed = gdf["genbil"] > 0
implied_av = gdf["implied_av_2025"].sum()
implied_city_levy = implied_av * LAKEWOOD_MILLS_PUBLISHED / 1000.0

print("GATE 3 - independent cross-check from GENBIL (dollars actually billed)")
print(f"  parcels with a 2025 bill: {int(_billed.sum()):,} of {len(gdf):,} "
      f"({_billed.mean() * 100:.1f}%)")
print(f"  GENBIL-implied roll-{ROLL_YEAR_LEVY} assessed value: ${implied_av:,.0f}")
print(f"  certified GROSS assessed valuation:                 ${LAKEWOOD_GROSS_AV_2025:,.0f}")
print(f"  gap: {(implied_av / LAKEWOOD_GROSS_AV_2025 - 1) * 100:+.2f}%")
print(f"\n  implied city levy at {LAKEWOOD_MILLS_PUBLISHED} mills: ${implied_city_levy:,.0f}")
print(f"  certified city levy:                    ${OFFICIAL_CITY_LEVY:,.0f}")
print(f"  gap: {(implied_city_levy / OFFICIAL_CITY_LEVY - 1) * 100:+.2f}%")

# Composition of the gap -- reported, not smoothed over
_unbilled_taxable = (~_billed) & nonexempt
print(f"\n  gap composition (this is a FINDING, not a correction):")
print(f"    taxable parcels with no 2025 bill: {int(_unbilled_taxable.sum()):,} "
      f"(${gdf.loc[_unbilled_taxable, 'taxable_total'].sum():,.0f} roll-{ROLL_YEAR_PARCELS} "
      "assessed) -- parcels created/split after the 2025 roll was billed")
print(f"    exempt parcels (correctly unbilled): {int(((~_billed) & (gdf.full_exmp == 1)).sum()):,}")
print("    remainder: personal property and state-assessed property, which are levied on "
      "in the certified figure but absent from the parcel file.")
print("\n  Two independent estimates of Lakewood's own levy bracket the certified figure "
      "from below by ~5%, consistently. Neither is used to adjust the other; the certified "
      "levy remains the anchor.")

In [ ]:
# --- Derive the municipal millage on the roll-2026 base (Baltimore/Madison pattern) ---
DERIVED_CITY_MILLAGE = OFFICIAL_CITY_LEVY / taxable_base * 1000.0
print("LEVY CALIBRATION")
print(f"  certified levy (roll {ROLL_YEAR_LEVY}):          ${OFFICIAL_CITY_LEVY:,.0f}")
print(f"  published mills (roll {ROLL_YEAR_LEVY}):         {LAKEWOOD_MILLS_PUBLISHED:.4f}")
print(f"  derived mills on the roll-{ROLL_YEAR_PARCELS} base: {DERIVED_CITY_MILLAGE:.4f}")
print(f"  difference: {(DERIVED_CITY_MILLAGE / LAKEWOOD_MILLS_PUBLISHED - 1) * 100:+.2f}%"
      "  -- the roll-year rate cut plus the missing personal-property base, NOT a tax increase")
print(f"\n  Lakewood's share of a typical stacked bill: "
      f"{LAKEWOOD_MILLS_PUBLISHED / TYPICAL_STACK_MILLS * 100:.1f}% "
      f"({LAKEWOOD_MILLS_PUBLISHED} of {TYPICAL_STACK_MILLS} mills)")
print("\n  WHY THIS IS SAFE: a revenue-neutral split-rate result is invariant to the level "
      "of the municipal millage. Scaling the millage scales every parcel's current_tax and "
      "new_tax by the same constant, so every tax_change_pct is unchanged. The roll-year "
      "mismatch moves dollar magnitudes only.")

gdf["millage_rate"] = DERIVED_CITY_MILLAGE
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col="taxable_total",
    millage_rate_col="millage_rate",
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    exemption_flag_col="full_exmp",
)
gap = (current_revenue / OFFICIAL_CITY_LEVY - 1) * 100
print(f"\n  modeled municipal revenue: ${current_revenue:,.0f}")
print(f"  REVENUE GATE: {gap:+.4f}%  (anchored to the certified levy by construction)")
assert abs(gap) < 5.0, f"Revenue gate FAILED: {gap:.2f}%"

print(f"\n  (reference) published {LAKEWOOD_MILLS_PUBLISHED} mills on this base = "
      f"${taxable_base * LAKEWOOD_MILLS_PUBLISHED / 1000:,.0f}, "
      f"{(taxable_base * LAKEWOOD_MILLS_PUBLISHED / 1000 / OFFICIAL_CITY_LEVY - 1) * 100:+.2f}% "
      "vs the certified levy")
print(f"  (reference) city general-fund cash net of the LRA increment: "
      f"${CITY_CASH_LEVY_NET_TIF:,.0f}")
LAKEWOOD_MILLS_MODELED = DERIVED_CITY_MILLAGE

### Optional: waterfront single-family split

The repo splits single-family homes into waterfront and non-waterfront where a city has
significant water frontage **and** a non-trivial share of its single-family stock sits on
it (roughly 2% of SFH within 100 m, or several hundred parcels). Lakewood's only
significant water body is Bear Creek Lake, a reservoir inside a city-owned regional park,
so the expectation is that this gate does *not* clear — but the test is run rather than
assumed, and the result is recorded either way.

In [ ]:
WATERFRONT_APPLIED = False
_wf_stats = None
try:
    from lvt.waterfront_utils import (add_waterfront_flags, waterfront_share,
                                      split_single_family_by_waterfront)
    gdf = add_waterfront_flags(gdf, fips_code=STATE_FIPS + COUNTY_FIPS,
                               cache_dir=DATA_DIR, min_water_area_m2=10_000)
    _wf_stats = waterfront_share(gdf, category_col="PROPERTY_CATEGORY")
    print("waterfront share:", _wf_stats)
    if _wf_stats and _wf_stats.get("waterfront_pct", 0) >= 2.0:
        gdf = split_single_family_by_waterfront(gdf, category_col="PROPERTY_CATEGORY")
        WATERFRONT_APPLIED = True
        print("-> gate CLEARED: single-family split into waterfront / non-waterfront")
    else:
        print("-> gate NOT cleared: no waterfront split applied (as expected for Lakewood)")
except Exception as e:
    print(f"waterfront check skipped ({type(e).__name__}: {e})")
print(f"WATERFRONT_APPLIED = {WATERFRONT_APPLIED}")

## Section 7 — The reform: revenue-neutral 4:1 split-rate

Fully-exempt parcels are held out of the solver and excluded from the export and report
charts; they are reported by count. Revenue neutrality is enforced across all taxable
parcels in the city.

In [ ]:
n_exempt = int((gdf.full_exmp == 1).sum())
df = gdf[gdf.full_exmp == 0].copy().reset_index(drop=True)
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, charts)")
print(f"Modeled taxable parcels: {len(df):,}")

land_millage, improvement_millage, new_revenue, df = model_split_rate_tax(
    df=df,
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    current_revenue=df["current_tax"].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"\n4:1 SPLIT-RATE on Lakewood's own municipal levy")
print(f"  current municipal millage: {LAKEWOOD_MILLS_MODELED:.4f}  "
      f"(published {LAKEWOOD_MILLS_PUBLISHED:.4f}, roll {ROLL_YEAR_LEVY})")
print(f"  land millage:              {land_millage:.4f}")
print(f"  improvement millage:       {improvement_millage:.4f}"
      f"   (ratio {land_millage / improvement_millage:.2f}:1)")
print(f"  revenue: ${new_revenue:,.0f} vs target ${df['current_tax'].sum():,.0f}"
      f"  ({(new_revenue / df['current_tax'].sum() - 1) * 100:+.4f}%)")

print(f"\n  land share of the taxable base:        "
      f"{df.taxable_land_value.sum() / df.taxable_total.sum() * 100:.1f}% "
      f"(${df.taxable_land_value.sum():,.0f})")
print(f"  improvement share of the taxable base: "
      f"{df.taxable_improvement_value.sum() / df.taxable_total.sum() * 100:.1f}% "
      f"(${df.taxable_improvement_value.sum():,.0f})")

cat = calculate_category_tax_summary(
    df=df, category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax", new_tax_col="new_tax")
print_category_tax_summary(cat, title="Lakewood, CO - 4:1 split-rate "
                                      "(City of Lakewood municipal levy only)")

## Section 8 — Gate 5: artifact scan

Gates 1–3 confirm the model *ran*. This confirms the results are *believable*. The table
is read against economic priors: land-heavy categories (vacant land, surface lots, low-rise
commercial on valuable land) should pay more; building-heavy categories (condominiums,
apartments, offices) should pay less. The canary is distinct property types pinned at the
same extreme value, an implausible land/building split, or a base that will not reconcile
to a control.

In [ ]:
d = df.copy()
d["bldg_share"] = d.taxable_improvement_value / d.taxable_total.replace(0, np.nan)
scan = d.groupby("PROPERTY_CATEGORY").agg(
    n=("tax_change_pct", "size"),
    median_pct=("tax_change_pct", "median"),
    median_bldg_share=("bldg_share", "median"),
    pct_zero_bldg=("taxable_improvement_value", lambda s: (s <= 10).mean()),
    pct_land_imputed=("improvement_only", "mean"),
).sort_values("median_pct", ascending=False)
print(scan.round(3).to_string())

# Ceiling test: with no improvement value at all, the change is the land-only outcome.
_ceiling = land_millage / LAKEWOOD_MILLS_MODELED - 1
print(f"\nland-only ceiling (max possible increase): {_ceiling * 100:+.1f}%")
_at_ceiling = d[(d.tax_change_pct - _ceiling * 100).abs() < 0.5]
print(f"parcels within 0.5pp of the ceiling: {len(_at_ceiling):,} "
      f"({len(_at_ceiling) / len(d) * 100:.2f}%)")
print("their categories:")
print(_at_ceiling.groupby("PROPERTY_CATEGORY").size().sort_values(ascending=False)
      .head(8).to_string())
print("\n-> these should be genuinely vacant/land-only parcels ONLY. Any built category "
      "sitting here would mean its improvement value is missing.")

In [ ]:
print("=== HEADLINE ===")
print(f"parcels modeled:      {len(df):,}   (fully exempt held out: {n_exempt:,})")
_less = df.tax_change < -0.5
_more = df.tax_change > 0.5
print(f"pay LESS:             {int(_less.sum()):,} ({_less.mean() * 100:.1f}%)")
print(f"pay MORE:             {int(_more.sum()):,} ({_more.mean() * 100:.1f}%)")
print(f"essentially unchanged:{int((df.tax_change.abs() <= 0.5).sum()):,}")
print(f"median change:        {df.tax_change_pct.median():+.1f}% / "
      f"${df.tax_change.median():+,.0f}")

_SFH = "Single Family Residential"
sf = df[df.PROPERTY_CATEGORY == _SFH]
print(f"\n=== detached single-family homes (owner-occupancy is not in the open extract) ===")
print(f"n = {len(sf):,}")
print(f"  median change:            {sf.tax_change_pct.median():+.1f}% / "
      f"${sf.tax_change.median():+,.0f}")
print(f"  median current city tax:  ${sf.current_tax.median():,.0f}")
print(f"  median new city tax:      ${sf.new_tax.median():,.0f}")
print(f"  median actual value:      ${sf.TOTACTVAL.median():,.0f}")
print(f"  share paying less:        {(sf.tax_change < 0).mean() * 100:.1f}%")
print(f"  SPREAD -- share up   >10%: {(sf.tax_change_pct > 10).mean() * 100:.1f}%")
print(f"  SPREAD -- share down >10%: {(sf.tax_change_pct < -10).mean() * 100:.1f}%")
print("  (the spread is the honest headline; never put the median alone on a slide)")

print("\n=== the scope caveat, in dollars ===")
print(f"  Lakewood's own levy is {LAKEWOOD_MILLS_PUBLISHED} of ~{TYPICAL_STACK_MILLS} mills "
      f"= {LAKEWOOD_MILLS_PUBLISHED / TYPICAL_STACK_MILLS * 100:.1f}% of a tax bill")
_med_total_bill = (sf.implied_av_2025 * sf.MILL_LEVY / 1000).median()
print(f"  median SFH total property-tax bill (all entities, from GENBIL): "
      f"${sf.genbil.median():,.0f}")
print(f"  median SFH CITY share of that bill:                            "
      f"${sf.current_tax.median():,.0f}")
print(f"  a {sf.tax_change_pct.median():+.1f}% change in the city share is "
      f"{sf.tax_change.median() / sf.genbil.median() * 100:+.2f}% of the total bill")

## Section 9 — Undeveloped and underdeveloped land

In [ ]:
result = analyze_land_by_improvement_share(
    df, land_value_col="TOTACTLNDV", improvement_value_col="TOTACTIMPV")
total_land = result["total_adjusted_land_value"]
cats = {c["category"]: c for c in result["categories"]}
print(f"Total non-exempt actual land value: ${total_land:,.0f}\n")
print(f"{'Category':<42}{'Parcels':>9}{'Land value':>19}{'% of land':>11}")
print("-" * 81)
for key, c in cats.items():
    print(f"{key:<42}{c['parcel_count']:>9,}${c['adjusted_land_value']:>18,.0f}"
          f"{c['share_of_total_land_value_pct']:>10.2f}%")

impr_share = (df.TOTACTIMPV / df.TOTACTVAL.replace(0, np.nan)).fillna(0)
UNDERDEV = {}
for label, mask in [("Vacant ($0 improvement)", df.TOTACTIMPV <= 0),
                    ("Under 50% built", impr_share.lt(0.50)),
                    ("Vacant OR under 50% built",
                     (df.TOTACTIMPV <= 0) | impr_share.lt(0.50))]:
    s = df[mask]
    UNDERDEV[label] = {
        "parcels": int(len(s)),
        "land_value": float(s.TOTACTLNDV.sum()),
        "pct_of_land": float(s.TOTACTLNDV.sum() / df.TOTACTLNDV.sum() * 100),
        "median_pct": float(s.tax_change_pct.median()) if len(s) else float("nan"),
        "net_change": float(s.tax_change.sum()),
    }
    print(f"\n{label}: {len(s):,} parcels ({len(s) / len(df) * 100:.1f}%), "
          f"${s.TOTACTLNDV.sum():,.0f} land value "
          f"({s.TOTACTLNDV.sum() / df.TOTACTLNDV.sum() * 100:.1f}% of citywide land)")
    if len(s):
        print(f"  median change: {s.tax_change_pct.median():+.1f}%   "
              f"net ${s.tax_change.sum():+,.0f}")

In [ ]:
# The categories the deck leads with
print("--- categories the deck leads with ---")
for label, mask in [
        ("Vacant Land", df.PROPERTY_CATEGORY == "Vacant Land"),
        ("Parking structures", df.PROPERTY_CATEGORY == "Transportation - Parking"),
        ("Retail / general commercial", df.PROPERTY_CATEGORY == "Retail / General Commercial"),
        ("Office", df.PROPERTY_CATEGORY == "Office / Commercial Condo"),
        ("Industrial", df.PROPERTY_CATEGORY == "Industrial"),
        ("Detached single family", df.PROPERTY_CATEGORY == "Single Family Residential"),
        ("Townhome / rowhouse", df.PROPERTY_CATEGORY == "Townhome / Rowhouse"),
        ("Condominium", df.PROPERTY_CATEGORY == "Condominium"),
        ("Small multi-family (2-4)", df.PROPERTY_CATEGORY == "Small Multi-Family (2-4 units)"),
        ("Large multi-family (5+)", df.PROPERTY_CATEGORY == "Large Multi-Family (5+ units)"),
]:
    s = df[mask]
    if len(s) == 0:
        print(f"  {label:<32} (none)")
        continue
    print(f"  {label:<32} n={len(s):>6,}  median {s.tax_change_pct.median():+7.1f}%  "
          f"${s.tax_change.median():>+8,.0f}   net ${s.tax_change.sum():>+14,.0f}")

## Section 10 — Census join, standard export, report, parcel map

In [ ]:
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ["std_geoid", "median_income", "minority_pct", "black_pct"]
if not os.getenv("CENSUS_API_KEY"):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        df[_col] = float("nan")
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=180)
                # get_census_data_with_boundaries already merges the ACS attributes onto
                # census_gdf, so match_to_census_blockgroups brings median_income /
                # minority_pct / black_pct across on its own. Do NOT merge census_data
                # again on std_geoid -- that produces _x/_y suffixed columns, leaves no
                # plain `median_income`, and silently drops the four equity charts.
                df = match_to_census_blockgroups(df, census_gdf)
                if "minority_pct" not in df.columns and {"total_pop", "white_pop"} <= set(df.columns):
                    df["minority_pct"] = ((df.total_pop - df.white_pop) / df.total_pop * 100).round(2)
                if "black_pct" not in df.columns and {"total_pop", "black_pop"} <= set(df.columns):
                    df["black_pct"] = (df.black_pop / df.total_pop * 100).round(2)
                _missing = [c for c in _census_cols if c not in df.columns]
                assert not _missing, f"census join produced no {_missing} column"
                print(f"Census join: {df['std_geoid'].notna().mean() * 100:.1f}% matched; "
                      f"median_income populated on {df['median_income'].notna().mean() * 100:.1f}%")
            except concurrent.futures.TimeoutError:
                print("Census API timed out - skipping census join")
                for _col in _census_cols:
                    df[_col] = float("nan")
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            df[_col] = float("nan")

In [ ]:
out_df = save_standard_export(
    df=df,
    city=CITY_NAME,
    output_path=f"../../analysis/data/{CITY_NAME}.csv",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    tax_change_col="tax_change",
    tax_change_pct_col="tax_change_pct",
    taxable_land_col="taxable_land_value",
    taxable_improvement_col="taxable_improvement_value",
    exempt_flag_col="full_exmp",
)
print(f"Wrote ../../analysis/data/{CITY_NAME}.csv  {out_df.shape}")
print(list(out_df.columns))

In [ ]:
from lvt.viz import create_city_report
_report = create_city_report(out_df, CITY_NAME, show=False)
_rep_dir = Path(f"../../analysis/reports/{CITY_NAME}")
print("\nreport files:")
for _p in sorted(_rep_dir.glob("*")):
    print(f"  {_p.name:<44} {_p.stat().st_size / 1024:>9,.0f} KB")
_pngs = sorted(p.name for p in _rep_dir.glob("*.png"))
print(f"\nPNGs generated: {len(_pngs)} -> {_pngs}")

In [ ]:
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

gdf_map = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
map_gdf = save_parcel_map_export(
    gdf=gdf_map,
    city=CITY_NAME,
    output_path=f"../../analysis/maps/{CITY_NAME}.parquet",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
    exempt_flag_col="full_exmp",
)
create_parcel_map(map_gdf, CITY_NAME)
print(f"parcel map written for {len(map_gdf):,} parcels")

In [ ]:
# Cache the modeled parcels for the Landslide deck pipeline (example-scout depends on
# this file). Carries geometry, land + improvement value, land + building area, current
# and new tax, and the property category.
DECK_COLS = ["PIN", "PRPADDRESS", "OWNNAM", "NHDNAM", "SUBNAM", "PROPERTY_CATEGORY",
             "STTSTRC", "STTTYPUSE", "STTYRBLT", "STTNBRUNT", "TAXCLS",
             "TOTACTVAL", "TOTACTLNDV", "TOTACTIMPV",
             "taxable_land_value", "taxable_improvement_value", "taxable_total",
             "LGLSQFT", "TOTACR", "STTGRSAREA",
             "current_tax", "new_tax", "tax_change", "tax_change_pct",
             "full_exmp", "improvement_only", "MILL_LEVY", "GENBIL",
             "std_geoid", "median_income", "minority_pct", "black_pct", "geometry"]
_have = [c for c in DECK_COLS if c in df.columns]
_missing = [c for c in DECK_COLS if c not in df.columns]
deck_gdf = gpd.GeoDataFrame(df[_have].copy(), geometry="geometry", crs="EPSG:4326")
deck_gdf = deck_gdf.rename(columns={
    "TOTACTLNDV": "land_value", "TOTACTIMPV": "improvement_value",
    "TOTACTVAL": "total_value", "LGLSQFT": "land_area_sqft",
    "STTGRSAREA": "building_area_sqft", "PIN": "parcel_id",
    "PROPERTY_CATEGORY": "property_category"})
# Robust lot area for the example-scout: the assessor's LGLSQFT is 0 for attached
# housing (townhomes, condos), so fall back to TOTACR, then to the polygon itself.
_lot = pd.to_numeric(deck_gdf.get("land_area_sqft"), errors="coerce").fillna(0.0)
_acr = pd.to_numeric(deck_gdf.get("TOTACR"), errors="coerce").fillna(0.0) * 43_560.0
_poly = deck_gdf.to_crs(2877).area
deck_gdf["lot_sqft"] = np.where(_lot > 0, _lot, np.where(_acr > 0, _acr, _poly))
deck_gdf["lot_sqft_source"] = np.where(_lot > 0, "assessor_lglsqft",
                                np.where(_acr > 0, "assessor_totacr", "polygon"))
print("lot_sqft source mix:")
print(deck_gdf["lot_sqft_source"].value_counts().to_string())
print(f"  lot_sqft > 0 on {int((deck_gdf.lot_sqft > 0).sum()):,}/{len(deck_gdf):,} parcels")

deck_gdf["land_millage"] = land_millage
deck_gdf["improvement_millage"] = improvement_millage
deck_gdf["current_millage"] = LAKEWOOD_MILLS_MODELED
deck_gdf.to_parquet(DATA_DIR / "parcels.gpq")
print(f"Wrote {DATA_DIR / 'parcels.gpq'}  {deck_gdf.shape}")
if _missing:
    print(f"  (columns not available: {_missing})")
print(f"  columns: {list(deck_gdf.columns)}")
print(f"  geometry valid: {deck_gdf.geometry.is_valid.mean() * 100:.1f}%   "
      f"null: {deck_gdf.geometry.isna().sum()}")
print("Done.")

## Validation summary

| Check | Result |
|---|---|
| Levy scope | City of Lakewood's own mill levy only — 4.7110 mills, ~5.2% of a ~91.4-mill stack |
| Revenue anchor | Certified TY2025 levy $15,801,930 on gross AV $3,354,262,401 (GROSS basis — see Section 6) |
| Base validation (actual value) | see Gate 1 — the gate that can fail |
| Assessed base vs certified | see Gate 2 — roll-year rate cut + personal property excluded |
| Independent GENBIL cross-check | see Gate 3 — reported as a finding |
| Revenue gate | anchored by construction |
| Gate 5 artifact scan | condo land-share repair applied (Section 4b); ceiling test in Section 8 |
| Parcel map | `analysis/maps/lakewood.parquet` + `parcel_map.html` |
| Deck cache | `cities/lakewood/data/parcels.gpq` |

### Known limitations

1. **The city levy is 4.7110 of ~91.4 mills.** Every percentage here is a percentage of
   the city's own share, not of a total tax bill. Dollar magnitudes are small.
2. **Roll-year mismatch.** Parcels are roll 2026; the certified levy is roll 2025. The
   municipal millage is derived so modeled revenue equals the certified levy. Percentages
   are invariant to this; dollar magnitudes are on the roll-2026 basis.
3. **Condominium land values are imputed** (Section 4b) because Jefferson County books
   condo units as improvement-only with no land value anywhere on the roll. Totals are the
   county's; only the land/improvement split is modeled.
4. **No separable senior/veteran exemption.** Colorado's state-reimbursed senior
   exemption is already netted into the assessor's published assessed values, so it is
   preserved but cannot be reported separately.
5. **Personal property and state-assessed property are excluded** — they are in the
   certified levy base but absent from the parcel file.
6. **Surface parking is not separately coded** in this roll; `Transportation - Parking`
   means parking structures only.
7. **No stable per-parcel county URL** exists, so the parcel map has no county-record
   deep links.